In [ ]:
get_ipython().run_line_magic("pip", "install -q transformers datasets peft trl accelerate")

In [ ]:
from pathlib import Path

files = list(Path("/kaggle/input").rglob("ml_tutor_golden_preference_1184.jsonl"))

print(files)

In [ ]:
DATA_PATH = str(files[0])

print("Dataset path:")
print(DATA_PATH)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=DATA_PATH,
    split="train"
)

print(dataset)

In [ ]:
print("Total Columns:", dataset.column_names)
print("Total preference pairs:", len(dataset))

In [ ]:
example = dataset[0]

print("ID: ", example["id"])

print("\nTOPIC: ", example["topic"])

print("\nPROMPT:")
print(example["prompt"])

print("\nCHOSEN:")
print(example["chosen"])

print("\nREJECTED:")
print(example["rejected"])

print("\nWHY REJECTED:")
print(example["rejection_reason"])

In [ ]:
for i in range(3):

    row = dataset[i]

    print("\n" + "=" * 100)
    print(f"EXAMPLE {i + 1}")
    print("=" * 100)

    print("\nTopic: ", row["topic"])

    print("\nUser: ", row["prompt"][-1]["content"])

    print("\n✅ Chosen:")
    print(row["chosen"][0]["content"])

    print("\n❌ Rejected:")
    print(row["rejected"][0]["content"])

    print("\nReason: ", row["rejection_reason"])

In [ ]:
# BASIC DATASET QUALITY CHECKS

print("Total rows:", len(dataset))

# Empty fields
empty_prompt = 0
empty_chosen = 0
empty_rejected = 0

# Chosen and rejected accidentally same?
same_answers = 0

for row in dataset:

    if not row["prompt"]:
        empty_prompt += 1

    if not row["chosen"]:
        empty_chosen += 1

    if not row["rejected"]:
        empty_rejected += 1

    chosen_text = row["chosen"][0]["content"].strip()
    rejected_text = row["rejected"][0]["content"].strip()

    if chosen_text == rejected_text:
        same_answers += 1

In [ ]:
print("Empty prompts:", empty_prompt)
print("Empty chosen:", empty_chosen)
print("Empty rejected:", empty_rejected)
print("Chosen == Rejected:", same_answers)

In [ ]:
# DUPLICATE PROMPTS CHECK

import pandas as pd

df = dataset.to_pandas()

# Extract only the user question
df["user_prompt"] = df["prompt"].apply(lambda x: x[-1]["content"].strip().lower())

duplicate_count = df["user_prompt"].duplicated().sum()

print("Duplicate prompts:", duplicate_count)

In [ ]:
# CHOSEN / REJECTED ANSWERS LENGTHS CHECK

df["chosen_words"] = df["chosen"].apply(lambda x: len(x[0]["content"].split()))

df["rejected_words"] = df["rejected"].apply(lambda x: len(x[0]["content"].split()))

print("Chosen answer length: ")
print(df["chosen_words"].describe())

print("\nRejected answer length: ")
print(df["rejected_words"].describe())

In [ ]:
print("CHAPTERS:")
print(df["chapter"].value_counts())

print("\nPAIR TYPES:")
print(df["pair_type"].value_counts())

In [ ]:
print(df["topic"].value_counts().head(20))

In [ ]:
# ANSWER KITNE SIMILAR HAIN

from difflib import SequenceMatcher

def similarity(row):

    chosen = row["chosen"][0]["content"]
    rejected = row["rejected"][0]["content"]

    return SequenceMatcher(None, chosen, rejected).ratio()

In [ ]:
similarities = [similarity(row) for row in dataset]

print(pd.Series(similarities).describe())

In [ ]:
df["similarity"] = similarities

In [ ]:
def show_pairs(rows, title):

    print("\n" + "=" * 150)
    print(title)
    print("=" * 150)

    for _, row in rows.iterrows():

        print("\nTOPIC: ", row["topic"])

        print("\nQUESTION: ", row["prompt"][-1]["content"])

        print("\n✅ CHOSEN:")
        print(row["chosen"][0]["content"])

        print("\n❌ REJECTED:")
        print(row["rejected"][0]["content"])

        print("\nSIMILARITY: ", round(row["similarity"], 3))

        print("\n" + "-" * 80)

In [ ]:
show_pairs(df.nlargest(5, "similarity"), "5 HIGHEST SIMILARITY PAIRS")

show_pairs(df.nsmallest(5, "similarity"), "5 LOWEST SIMILARITY PAIRS")

In [ ]:
BOILERPLATE_PATTERNS = [
    "If it is tunable, choose it with validation or cross-validation rather than the final test set.",
    "This is why the decision should be checked with held-out validation rather than assumed from training behavior alone.",
    "A reliable workflow checks the choice on held-out data.",
    "Keep any learned or tunable choice inside the training and validation process.",
    "For the most reliable choice, select the setting that gives the best score on the final test set.",
    "In practice this means training performance is usually enough to judge it.",
    "If training performance keeps improving, a validation drop can usually be ignored."
]

def count_boilerplate(text):
    return sum(pattern in text for pattern in BOILERPLATE_PATTERNS)
    

df["chosen_boilerplate"] = df["chosen"].apply(lambda x: count_boilerplate(x[0]["content"]))

df["rejected_boilerplate"] = df["rejected"].apply(lambda x: count_boilerplate(x[0]["content"]))

In [ ]:
flagged = df[(df["chosen_boilerplate"] > 0) | (df["rejected_boilerplate"] > 0)]

print("Flagged rows:", len(flagged))

print(flagged["pair_type"].value_counts())

In [ ]:
mask = ((df["chosen_boilerplate"] > 0) | (df["rejected_boilerplate"] > 0))

rewrite_df = df[mask].copy()
clean_df = df[~mask].copy()

print("Clean pairs:", len(clean_df))
print("Pairs to rewrite:", len(rewrite_df))
print("Total:", len(clean_df) + len(rewrite_df))

print("\nPairs to rewrite by type:")
print(rewrite_df["pair_type"].value_counts())

In [ ]:
# CHECKING ALL ROWS EQUALLY DISTRIBUTED  IN 4 TOPICS

topic_counts = rewrite_df["topic"].value_counts()

print(topic_counts.describe())

print("\nFirst 20 topics:")
print(topic_counts.head(20))

In [ ]:
# CHECKING 3 RANDOM PAIRS OF EVERY 4 TOPICS

problem_types = ["limitation", "parameter", "what_if", "workflow"]

for pair_type in problem_types:

    print("\n" + "=" * 100)
    print("PAIR TYPE:", pair_type.upper())
    print("=" * 100)

    subset = rewrite_df[rewrite_df["pair_type"] == pair_type]

    samples = subset.sample(3, random_state=42)

    for _, row in samples.iterrows():

        print("\nTOPIC: ", row["topic"])

        print("\nQUESTION: ", row["prompt"][-1]["content"])

        print("\n✅ CHOSEN:")
        print(row["chosen"][0]["content"])

        print("\n❌ REJECTED:")
        print(row["rejected"][0]["content"])

        print("\n" + "-" * 100)

In [ ]:
from copy import deepcopy

In [ ]:
# Remove the two weak/template-heavy pair types

DROP_TYPES = {"parameter", "workflow",}

final_df = df[~df["pair_type"].isin(DROP_TYPES)].copy()

In [ ]:
# capitalize first letter after cleaning

def capitalize_first(text):
    text = text.strip()

    if not text:
        return text

    return text[0].upper() + text[1:]

In [ ]:
# Clean chosen/rejected answers

def clean_answer(messages, pair_type, side):

    # Make a safe copy of:
    # [{"role": "assistant", "content": "..."}]
    messages = deepcopy(messages)

    text = messages[0]["content"].strip()
    

    if pair_type == "limitation":

        if side == "chosen":

            text = text.replace(
                " This is why the decision should be checked "
                "with held-out validation rather than assumed "
                "from training behavior alone.",
                ""
            )

        else:

            text = text.replace("The main limitation is usually minor because ", "")

            text = text.replace(" In practice this means training performance is usually enough to judge it.", "")

    elif pair_type == "what_if":

        if side == "chosen":

            text = text.replace("A common mistake is to ignore this risk: ", "")

            text = text.replace(" A reliable workflow checks the choice on held-out data.", "")

        else:

            text = text.replace("The main risk is small. ", "")

            text = text.replace(" If training performance keeps improving, a validation drop can usually be ignored.", "")

    text = capitalize_first(text)

    messages[0]["content"] = text

    return messages

In [ ]:
# Apply cleaning only where needed

def clean_row(row):

    pair_type = row["pair_type"]

    if pair_type in {"limitation", "what_if"}:

        row["chosen"] = clean_answer(row["chosen"], pair_type, "chosen")

        row["rejected"] = clean_answer(row["rejected"], pair_type, "rejected")

    return row

In [ ]:
final_df = final_df.apply(clean_row, axis=1).reset_index(drop=True)

In [ ]:
print("Original pairs:", len(df))
print("Final pairs:", len(final_df))

print("\nPair types:")
print(final_df["pair_type"].value_counts())

In [ ]:
print("Final pairs:", len(final_df))

print("\nRemaining pair types:\n", final_df["pair_type"].value_counts())

In [ ]:
import numpy as np

def normalize(value):

    if isinstance(value, np.ndarray):
        value = value.tolist()

    if isinstance(value, dict):
        value = [value]

    return value

In [ ]:
final_df["prompt"] = final_df["prompt"].apply(normalize)
final_df["chosen"] = final_df["chosen"].apply(normalize)
final_df["rejected"] = final_df["rejected"].apply(normalize)

In [ ]:
row = final_df.iloc[0]

print(type(row["prompt"]))
print(type(row["chosen"]))
print(type(row["rejected"]))

print(row["chosen"][0]["content"])

In [ ]:
sample = final_df[final_df["pair_type"].isin(["limitation", "what_if"])].sample(5,random_state=42)

In [ ]:
for _, row in sample.iterrows():

    print("\n" + "=" * 80)

    print("TYPE:", row["pair_type"])
    print("TOPIC:", row["topic"])

    print("\nQUESTION:")
    print(row["prompt"][-1]["content"])

    print("\n✅ CHOSEN:")
    print(row["chosen"][0]["content"])

    print("\n❌ REJECTED:")
    print(row["rejected"][0]["content"])

In [ ]:
OUTPUT_PATH = "/kaggle/working/ml_tutor_preference_clean_1036.jsonl"

final_df.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(final_df))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
def split_by_topic(data, test_size, seed):

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=seed
    )

    left_idx, right_idx = next(splitter.split(data, groups=data["topic"]))

    left = data.iloc[left_idx].copy()
    right = data.iloc[right_idx].copy()

    return left, right

In [ ]:
SEED = 42

train_df, temp_df = split_by_topic(final_df, test_size=0.20, seed=SEED)

validation_df, test_df = split_by_topic(temp_df, test_size=0.50, seed=SEED + 1)

In [ ]:
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

print("\nTotal:", len(train_df) + len(validation_df) + len(test_df))

In [ ]:
# DATA LEAKAGE CHECK

train_topics = set(train_df["topic"])
val_topics = set(validation_df["topic"])
test_topics = set(test_df["topic"])

print("Train ∩ Validation:", len(train_topics & val_topics))

print("Train ∩ Test:", len(train_topics & test_topics))

print("Validation ∩ Test:", len(val_topics & test_topics))

In [ ]:
from datasets import Dataset, DatasetDict

In [ ]:
def to_hf_dataset(frame):
    
    return Dataset.from_pandas(
        
        frame[["prompt", "chosen", "rejected"]].reset_index(drop=True),
        
        preserve_index=False
    )

In [ ]:
splits = DatasetDict({
    "train": to_hf_dataset(train_df),
    "validation": to_hf_dataset(validation_df),
    "test": to_hf_dataset(test_df),
})

In [ ]:
print(splits)

In [ ]:
import transformers
import trl
import peft
import datasets

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_WRITE_TOKEN")

login(token=token)

In [ ]:
get_ipython().getoutput("pip uninstall -y torchao")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

LOCAL_BASE = "/kaggle/input/datasets/mohdmaaz036/qwen2-5-1-5b-instruct-base"
SFT_ADAPTER = "mohd-maaz/qwen-ml-tutor-final"

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_BASE,
    local_files_only=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_BASE,
    local_files_only=True
)

model = PeftModel.from_pretrained(
    base_model,
    SFT_ADAPTER,
    is_trainable=True
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
from copy import deepcopy

# SAVING SETTINGS OF CURRENT SFT LoRA
dpo_peft_config = deepcopy(model.peft_config["default"])

# MERGING LEARNING OF LoRA IN BASE MODEL WEIGHTS
model = model.merge_and_unload()

print(type(model))

In [ ]:
print("r:", dpo_peft_config.r)
print("alpha:", dpo_peft_config.lora_alpha)
print("dropout:", dpo_peft_config.lora_dropout)
print("target modules:", dpo_peft_config.target_modules)

In [ ]:
from trl import DPOConfig

dpo_args = DPOConfig(
    output_dir="/kaggle/working/qwen-ml-tutor-dpo",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    beta=0.1,

    max_length=512,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=10,

    fp16=True,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    save_total_limit=2,

    report_to="none",
)

In [ ]:
from trl import DPOTrainer

tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
trainer = DPOTrainer(
    model=model,
    
    ref_model=None,

    args=dpo_args,

    train_dataset=splits["train"],
    
    eval_dataset=splits["validation"],

    processing_class=tokenizer,

    peft_config=dpo_peft_config,
)

In [ ]:
trainer.model.print_trainable_parameters()

In [ ]:
print("Train after processing:", len(trainer.train_dataset))
print("Validation after processing:", len(trainer.eval_dataset))

In [ ]:
trainer.model.config.use_cache = False

train_result = trainer.train()

In [ ]:
test_metrics = trainer.evaluate(
    eval_dataset=splits["test"],
    metric_key_prefix="test"
)

test_metrics

In [ ]:
# Only DPO LoRA backup (~small)
DPO_ADAPTER_PATH = "/kaggle/working/qwen-ml-tutor-dpo-adapter"

trainer.save_model(DPO_ADAPTER_PATH)
tokenizer.save_pretrained(DPO_ADAPTER_PATH)

print("DPO adapter saved")

In [ ]:
final_model = trainer.model.merge_and_unload()

print(type(final_model))

In [ ]:
# Complete merged SFT + DPO model (~3 GB)
FINAL_MODEL_PATH = "/kaggle/working/qwen-ml-tutor-dpo-final"

final_model.config.use_cache = True

final_model.save_pretrained(
    FINAL_MODEL_PATH,
    safe_serialization=True
)

tokenizer.save_pretrained(FINAL_MODEL_PATH)

print("Final model saved:", FINAL_MODEL_PATH)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

RELOAD_PATH = "/kaggle/working/qwen-ml-tutor-dpo-final"

reload_tokenizer = AutoTokenizer.from_pretrained(RELOAD_PATH)

reload_model = AutoModelForCausalLM.from_pretrained(RELOAD_PATH)

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an expert machine learning tutor. Answer clearly and concisely in English."
    },
    {
        "role": "user",
        "content": "What is the difference between overfitting and underfitting?"
    }
]

inputs = reload_tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(reload_model.device)

outputs = reload_model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
    repetition_penalty=1.1
)

input_length = inputs["input_ids"].shape[-1]

answer = reload_tokenizer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True
)

print(answer)

In [ ]:
test_questions = [
    "What is the difference between precision and recall?",
    "Why is feature scaling important for k-nearest neighbors?",
    "What is the difference between Lasso and Ridge regression?",
    "Why should the test set not be used for hyperparameter tuning?",
    "What does PCA do in machine learning?"
]

for question in test_questions:

    messages = [
        {
            "role": "system",
            "content": "You are an expert machine learning tutor. Answer clearly and concisely in English."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = reload_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(reload_model.device)

    outputs = reload_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.1
    )

    input_length = inputs["input_ids"].shape[-1]

    answer = reload_tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

    print("\n" + "=" * 80)

In [ ]:
from huggingface_hub import create_repo

REPO_ID = "mohd-maaz/qwen-ml-tutor-dpo-final"

create_repo(
    repo_id=REPO_ID,
    private=False,
    exist_ok=True
)

print("Repository ready:", REPO_ID)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/kaggle/working/qwen-ml-tutor-dpo-final",
    repo_id="mohd-maaz/qwen-ml-tutor-dpo-final",
    repo_type="model"
)

print("Final model uploaded successfully")